In [5]:
import pandas as pd
import math, re, html

# ---- Robust CSV read ----
df = pd.read_csv(
    '../input_data/carolina_schedule.csv',
    encoding='utf-8-sig',        # strips BOM if present
    skip_blank_lines=True,
    dtype=str,                   # read everything as strings first
    keep_default_na=False        # don't auto-convert "nan"/"NaN" strings
)

# Strip whitespace from column names, drop fully empty extra columns
df.columns = df.columns.str.strip()
df = df.dropna(axis=1, how='all')

# If there are trailing-commas columns, they'll be named like 'Unnamed: N'
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

# Keep only the columns you expect (after normalizing their names)
expected = ['Day', 'Time (EDT)', 'Location', 'Event', 'Speaker']
missing = [c for c in expected if c not in df.columns]
if missing:
    print("WARNING: Missing expected columns:", missing)
df = df.reindex(columns=expected)


# Normalize cell strings: strip, convert empty/whitespace/'nan' to NA
df = df.map(lambda x: None if (x is None or str(x).strip().lower() in {'', 'nan', 'none', '--'}) else str(x).strip())

# ---- The rest (same as your builder) ----
def safe(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return '--'
    return html.escape(str(v), quote=True)

def day_key(s):
    m = re.search(r'(monday|tuesday|wednesday|thursday|friday)', str(s), re.I)
    return m.group(1).lower() if m else str(s).strip().lower()

order_keys   = ['monday','tuesday','wednesday','thursday','friday']
display_name = {k: k.capitalize() for k in order_keys}
day_class    = {
    'monday':    'day-observations',
    'tuesday':   'day-theory',
    'wednesday': 'day-sims',
    'thursday':  'day-hack',
    'friday':    'day-hack',
}
day_focus = {
    'monday':    'Observations',
    'tuesday':   'Modeling',
    'wednesday': 'Simulations',
    'thursday':  'Hack Day',
    'friday':    'Hack Day',
}

# derive present days
keys_in_csv = df['Day'].apply(day_key)
present = [k for k in order_keys if (keys_in_csv == k).any()]

table_headers = ['Time (EDT)', 'Location', 'Event', 'Speaker']
tables_html = []

for k in present:
    day_df = df[keys_in_csv == k]
    if day_df.empty:
        continue

    cls   = day_class.get(k, 'day-default')
    aid   = k
    title = display_name.get(k, k.capitalize())
    focus = day_focus.get(k, '')

    rows = []
    for _, r in day_df.iterrows():
        cells = ''.join(f"<td>{safe(r[col])}</td>" for col in table_headers)
        rows.append(f"<tr>{cells}</tr>")

    heading_html = html.escape(title)
    if focus:
        heading_html += f' — <span class="schedule-focus">Focus: {html.escape(focus)}</span>'

    table_html = f"""
<h2 id="{aid}" class="schedule-day {cls}">{heading_html}</h2>
<table class="schedule-table {cls}">
  <thead>
    <tr>
      {''.join(f'<th>{html.escape(h)}</th>' for h in table_headers)}
    </tr>
  </thead>
  <tbody>
    {''.join(rows)}
  </tbody>
</table>
""".strip()
    tables_html.append(table_html)

html_schedule = "\n\n".join(tables_html)
print(html_schedule)


<h2 id="monday" class="schedule-day day-observations">Monday — <span class="schedule-focus">Focus: Observations</span></h2>
<table class="schedule-table day-observations">
  <thead>
    <tr>
      <th>Time (EDT)</th><th>Location</th><th>Event</th><th>Speaker</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>09:00–09:30</td><td>Room 1 or Lounge</td><td>Coffee &amp; Arrival</td><td>N/A</td></tr><tr><td>09:30–10:00</td><td>Room 1</td><td>Welcome &amp; Overview</td><td>Sarcevic &amp; Troxel</td></tr><tr><td>10:00–11:00</td><td>Room 1</td><td>Brief Introductions</td><td>all</td></tr><tr><td>11:00–11:30</td><td>Room 1</td><td>Observations Overview</td><td>TBA</td></tr><tr><td>11:30–12:30</td><td>Room 1</td><td>Flash Talks: Observational Inputs</td><td>--</td></tr><tr><td>12:30–13:30</td><td>Hallway/Duke Campus</td><td>Lunch</td><td>N/A</td></tr><tr><td>13:30–14:30</td><td>Room 1</td><td>Project Pitch Block I</td><td>TBA</td></tr><tr><td>14:30–17:00</td><td>Room 2 / Lounge / Terrace</td><td>Dis